# 23 PEAD（決算発表後ドリフト）仮説検証（日本株）

| 項目 | 内容 |
|------|------|
| プロジェクトID | STOCK-PRED-001 |
| 入力データ | `data/equity_jp_ohlcv.parquet`, `data/macro.parquet`, `data/earnings_dates.parquet`（任意） |
| 作成者 | |
| 更新日 | 2026-07-08 |

## 検証する仮説

**帰無仮説（H₀）**: 決算発表後の超過リターン（CAR）は 0 に収束する（PEAD なし）  
**対立仮説（H₁）**: 決算発表後にサプライズ方向への持続的ドリフトが存在する（PEAD あり）  
**有意水準（α）**: 0.05

## 背景
PEAD（Post-Earnings Announcement Drift）は、決算発表後に市場の反応が段階的に織り込まれる現象。
戦略定義書（01_戦略定義書.md Phase 1）では「決算イベント特徴量（PEAD 対応）」を明示的に
要求しており、本ノートブックでは:
1. イベントスタディ（CAR 分析）で PEAD の存在を確認する
2. シグナルフィルタとして「決算直前・直後の3日間を除外する」実装の妥当性を検証する

## 結論メモ
<!-- 検証後に記入: p値・効果量・推奨実装 -->

---
## 0. 環境セットアップ

In [ ]:
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import spearmanr, ttest_1samp
import yaml

try:
    import japanize_matplotlib
except ImportError:
    print('japanize_matplotlib not available — Japanese labels may not render correctly')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 100)
%matplotlib inline

ALPHA = 0.05
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
N_BOOT = 1000  # ブートストラップ反復数

print('Setup OK')

In [ ]:
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR    = Path(cfg['paths']['data'])
FIGURES_DIR = Path(cfg['paths']['figures'])
TABLES_DIR  = Path(cfg['paths']['tables'])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

EQUITY_SYMBOLS = cfg['equity']['symbols']
DATA_START     = cfg['equity']['data_start']  # '2020-01-01'

print(f'Data dir    : {DATA_DIR.resolve()}')
print(f'Symbols     : {len(EQUITY_SYMBOLS)} 銘柄')
print(f'Data start  : {DATA_START}')

---
## 1. データ読み込み

In [ ]:
# ---- OHLCV ----
ohlcv_path = DATA_DIR / 'equity_jp_ohlcv.parquet'
if not ohlcv_path.exists():
    raise FileNotFoundError(
        f'{ohlcv_path} が見つかりません。\n'
        '先に 01_data/03_equity_data_collection.ipynb を実行してください。'
    )

df_ohlcv = pd.read_parquet(ohlcv_path)
df_ohlcv['date'] = pd.to_datetime(df_ohlcv['date'])
df_ohlcv = df_ohlcv.sort_values(['symbol', 'date']).reset_index(drop=True)

# 日次対数リターン
df_ohlcv['log_ret'] = df_ohlcv.groupby('symbol')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)

print(f'OHLCV: {df_ohlcv.shape[0]:,} rows  {df_ohlcv["symbol"].nunique()} symbols')
print(f'Period: {df_ohlcv["date"].min().date()} ~ {df_ohlcv["date"].max().date()}')
df_ohlcv.head(3)

In [ ]:
# ---- マクロ（ニッケイ225）: 市場調整済み異常リターン計算に使用 ----
macro_path = DATA_DIR / 'macro.parquet'
df_macro = None
mkt_ret_series = None

if macro_path.exists():
    df_macro = pd.read_parquet(macro_path)
    df_macro['date'] = pd.to_datetime(df_macro['date'])
    if 'nikkei225' in df_macro.columns:
        mkt_ret_series = (
            df_macro.set_index('date')['nikkei225']
            .apply(np.log).diff()
            .rename('mkt_log_ret')
        )
        print(f'Market return series loaded: {len(mkt_ret_series)} days')
    else:
        print('WARN: nikkei225 not in macro.parquet — will use raw returns instead of abnormal returns')
else:
    print('WARN: macro.parquet not found — will use raw returns instead of abnormal returns')

---
## 2. 決算日データの準備

`earnings_dates.parquet` が存在すれば読み込み、なければ yfinance の `quarterly_financials` インデックスから合成する。

In [ ]:
earnings_path = DATA_DIR / 'earnings_dates.parquet'
df_earnings = None

if earnings_path.exists():
    df_earnings = pd.read_parquet(earnings_path)
    df_earnings['earnings_date'] = pd.to_datetime(df_earnings['earnings_date'])
    print(f'earnings_dates.parquet loaded: {len(df_earnings)} records')
    print(df_earnings.head(3))
else:
    print('earnings_dates.parquet not found → yfinance から決算日を取得します')

In [ ]:
if df_earnings is None:
    try:
        import yfinance as yf
    except ImportError:
        raise ImportError('yfinance が必要です: pip install yfinance')

    earnings_records = []
    FILTER_START = pd.Timestamp('2020-01-01')
    FILTER_END   = pd.Timestamp('2025-12-31')

    print(f'yfinance から {len(EQUITY_SYMBOLS)} 銘柄の決算日を取得中...')
    for sym in EQUITY_SYMBOLS:
        try:
            ticker = yf.Ticker(sym)
            # quarterly_financials の列が決算発表に近い日付
            qf = ticker.quarterly_financials
            if qf is None or qf.empty:
                # earnings_dates を試みる
                ed = ticker.earnings_dates
                if ed is not None and not ed.empty:
                    dates = pd.DatetimeIndex(ed.index).tz_localize(None)
                else:
                    print(f'  SKIP {sym}: 決算日データなし')
                    continue
            else:
                # quarterly_financials の列名が決算期末日（発表日ではない可能性あり）
                dates = pd.DatetimeIndex(qf.columns).tz_localize(None)

            for d in dates:
                if FILTER_START <= d <= FILTER_END:
                    earnings_records.append({'symbol': sym, 'earnings_date': d})
        except Exception as e:
            print(f'  WARN {sym}: {e}')

    if earnings_records:
        df_earnings = pd.DataFrame(earnings_records)
        df_earnings = df_earnings.drop_duplicates().sort_values(['symbol', 'earnings_date'])
        df_earnings = df_earnings.reset_index(drop=True)
        # 保存
        df_earnings.to_parquet(earnings_path, index=False)
        print(f'\n保存: {earnings_path}')
        print(f'レコード数: {len(df_earnings)}')
        print(f'銘柄数: {df_earnings["symbol"].nunique()}')
        print(df_earnings.head(5))
    else:
        print('WARN: yfinance から決算日を取得できませんでした。合成データを生成します。')

In [ ]:
# フォールバック: 合成決算日（四半期ごとに概ね発表される想定）
if df_earnings is None or len(df_earnings) == 0:
    print('合成決算日を生成します（日本株: 3月決算 → 5月/8月/11月/2月 発表の概算）')

    # 日本株の代表的な決算発表月: 5月（本決算）、8月（1Q）、11月（2Q）、2月（3Q）
    ANNOUNCEMENT_MONTHS = [2, 5, 8, 11]
    synthetic_records = []

    for sym in EQUITY_SYMBOLS:
        sym_data = df_ohlcv[df_ohlcv['symbol'] == sym]['date'].sort_values()
        if len(sym_data) == 0:
            continue
        all_dates = set(sym_data.dt.date)
        # 2020〜2025年の各発表月の15〜25日あたりの営業日を決算日とする
        for year in range(2020, 2026):
            for month in ANNOUNCEMENT_MONTHS:
                # その月の15日に近い営業日を探す
                target = pd.Timestamp(year=year, month=month, day=15)
                for delta in range(0, 15):
                    candidate = target + timedelta(days=delta)
                    if candidate.date() in all_dates:
                        synthetic_records.append({'symbol': sym, 'earnings_date': candidate})
                        break

    df_earnings = pd.DataFrame(synthetic_records)
    df_earnings = df_earnings.drop_duplicates().sort_values(['symbol', 'earnings_date']).reset_index(drop=True)
    print(f'合成決算日: {len(df_earnings)} 件  {df_earnings["symbol"].nunique()} 銘柄')
    print(df_earnings.head(5))
    print('\nNOTE: 合成データのため結果は参考値。実際の決算発表日を用いることを強く推奨します。')

print(f'\n決算日データ確定: {len(df_earnings)} レコード')

---
## 3. イベントスタディのセットアップ

- **推定ウィンドウ**: T-120 〜 T-31（ベータ推定用）
- **イベントウィンドウ**: T-10 〜 T+30（観察対象）
- **異常リターン**: 銘柄日次リターン − 市場リターン（市場調整モデル）

In [ ]:
# 日付インデックスの整備
# 全銘柄の営業日カレンダーを作成
all_dates = sorted(df_ohlcv['date'].unique())
date_to_idx = {d: i for i, d in enumerate(all_dates)}

# 市場リターン（ニッケイ225）
if mkt_ret_series is not None:
    mkt_ret_map = mkt_ret_series.to_dict()
else:
    # フォールバック: 全銘柄の単純平均
    mkt_ret_map = (
        df_ohlcv.groupby('date')['log_ret'].mean().to_dict()
    )
    print('市場リターン: 全銘柄の平均リターンを代替として使用')

# 銘柄別の日次リターン辞書
sym_ret_map = (
    df_ohlcv.groupby('symbol')
    .apply(lambda g: g.set_index('date')['log_ret'].to_dict())
    .to_dict()
)

print(f'全営業日数: {len(all_dates)}')
print(f'イベントウィンドウ: T-10 〜 T+30')
print(f'推定ウィンドウ: T-120 〜 T-31')

---
## 4. CAR（累積異常リターン）計算

In [ ]:
# イベントウィンドウ設定
T_PRE   = -10   # イベント前日数
T_POST  = 30    # イベント後日数
T_EST_START = -120  # 推定ウィンドウ開始
T_EST_END   = -31   # 推定ウィンドウ終了


def get_date_at_offset(event_date, offset: int, all_dates: list):
    """event_date から offset 営業日後の日付を返す。"""
    if event_date not in date_to_idx:
        # 最も近い営業日を探す
        candidates = [d for d in all_dates if abs((d - event_date).days) <= 5]
        if not candidates:
            return None
        event_date = min(candidates, key=lambda d: abs((d - event_date).days))
    idx = date_to_idx.get(event_date)
    if idx is None:
        return None
    target_idx = idx + offset
    if 0 <= target_idx < len(all_dates):
        return all_dates[target_idx]
    return None


def calc_abnormal_return(sym, date, ret_map, mkt_map, beta=1.0):
    """個別銘柄の異常リターン = 銘柄リターン - beta * 市場リターン。"""
    sym_r = ret_map.get(date, np.nan)
    mkt_r = mkt_map.get(date, 0.0)
    if np.isnan(sym_r):
        return np.nan
    return sym_r - beta * mkt_r


# メインループ: 各イベントのARを計算
car_records = []

for _, row in df_earnings.iterrows():
    sym = row['symbol']
    event_date = pd.Timestamp(row['earnings_date'])

    if sym not in sym_ret_map:
        continue

    ret_map = sym_ret_map[sym]

    # 推定ウィンドウのベータ計算（OLS: y=sym_ret, x=mkt_ret）
    est_dates = [
        get_date_at_offset(event_date, t, all_dates)
        for t in range(T_EST_START, T_EST_END + 1)
    ]
    est_dates = [d for d in est_dates if d is not None]

    sym_rets_est = np.array([ret_map.get(d, np.nan) for d in est_dates])
    mkt_rets_est = np.array([mkt_ret_map.get(d, np.nan) for d in est_dates])
    valid_mask = ~(np.isnan(sym_rets_est) | np.isnan(mkt_rets_est))

    beta = 1.0  # デフォルト
    if valid_mask.sum() >= 20:
        try:
            import numpy.linalg as LA
            x = mkt_rets_est[valid_mask]
            y = sym_rets_est[valid_mask]
            # OLS: beta = cov(x,y) / var(x)
            beta = np.cov(x, y)[0, 1] / (np.var(x) + 1e-10)
            beta = np.clip(beta, 0.0, 3.0)  # 外れ値クリップ
        except Exception:
            beta = 1.0

    # イベントウィンドウの AR
    for t in range(T_PRE, T_POST + 1):
        d = get_date_at_offset(event_date, t, all_dates)
        if d is None:
            continue
        ar = calc_abnormal_return(sym, d, ret_map, mkt_ret_map, beta=beta)
        car_records.append({
            'symbol': sym,
            'earnings_date': event_date,
            't': t,
            'date': d,
            'ar': ar,
            'beta': beta,
        })

df_car = pd.DataFrame(car_records)
print(f'AR レコード数: {len(df_car):,}')
print(f'イベント数: {df_car["earnings_date"].nunique()}')
print(f'銘柄数: {df_car["symbol"].nunique()}')
df_car.head(5)

In [ ]:
# 各イベントの CAR を計算（イベント内で累積）
df_car = df_car.sort_values(['symbol', 'earnings_date', 't'])
df_car['car'] = df_car.groupby(['symbol', 'earnings_date'])['ar'].cumsum()

# t=0 を基準に CAR を正規化
car_at_0 = df_car[df_car['t'] == 0][['symbol', 'earnings_date', 'car']].rename(columns={'car': 'car_at_0'})
df_car = df_car.merge(car_at_0, on=['symbol', 'earnings_date'], how='left')
df_car['car_norm'] = df_car['car'] - df_car['car_at_0'].fillna(0)

# 平均 CAR by t
mean_car = df_car.groupby('t')['ar'].agg(['mean', 'std', 'count']).reset_index()
mean_car['se'] = mean_car['std'] / np.sqrt(mean_car['count'])
mean_car['cum_ar'] = mean_car['mean'].cumsum()

print('平均異常リターン (t=-5 〜 +10):')
print(mean_car[mean_car['t'].between(-5, 10)].to_string(index=False))

In [ ]:
# CAR チャート: 全イベント平均
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 日次 AR
ax = axes[0]
bar_colors = ['#2ca02c' if v > 0 else '#d62728' for v in mean_car['mean']]
ax.bar(mean_car['t'], mean_car['mean'] * 100, color=bar_colors, alpha=0.7, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', label='決算発表日 (T=0)')
ax.set_xlabel('イベント日 (T)')
ax.set_ylabel('平均異常リターン (%)')
ax.set_title('日次平均異常リターン')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# CAR（累積）
ax2 = axes[1]
ax2.plot(mean_car['t'], mean_car['cum_ar'] * 100, 'b-', linewidth=2, label='CAR (全イベント)')
ax2.fill_between(
    mean_car['t'],
    (mean_car['cum_ar'] - 1.96 * mean_car['se'].cumsum()) * 100,
    (mean_car['cum_ar'] + 1.96 * mean_car['se'].cumsum()) * 100,
    alpha=0.2, color='blue', label='95% CI'
)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(0, color='black', linewidth=1.5, linestyle='--', label='T=0')
ax2.axvspan(0, 10, alpha=0.05, color='green', label='PEAD ウィンドウ (T=0〜+10)')
ax2.set_xlabel('イベント日 (T)')
ax2.set_ylabel('CAR (%)')
ax2.set_title('累積異常リターン (CAR)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.suptitle('PEAD イベントスタディ（全イベント平均）', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_pead_car_all.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. サプライズ分位別 CAR 分析

In [ ]:
# サプライズプロキシ: 決算前5日のリターンを「市場期待」として使用
# T-5〜T-1 のCAR = 事前期待の代理変数

surprise_records = []
for (sym, ed), grp in df_car.groupby(['symbol', 'earnings_date']):
    pre = grp[(grp['t'] >= -5) & (grp['t'] <= -1)]['ar'].sum()
    post_early = grp[(grp['t'] >= 1) & (grp['t'] <= 10)]['ar'].sum()  # PEAD 初期
    post_late = grp[(grp['t'] >= 11) & (grp['t'] <= 30)]['ar'].sum()  # PEAD 後期
    day0_ar = grp[grp['t'] == 0]['ar'].values
    day0 = day0_ar[0] if len(day0_ar) > 0 else np.nan
    surprise_records.append({
        'symbol': sym,
        'earnings_date': ed,
        'pre_car_5d': pre,          # サプライズプロキシ
        'day0_ar': day0,            # 発表当日の反応
        'car_t1_t10': post_early,   # PEAD 初期 (T+1〜T+10)
        'car_t11_t30': post_late,   # PEAD 後期 (T+11〜T+30)
    })

df_surprise = pd.DataFrame(surprise_records).dropna(subset=['pre_car_5d', 'car_t1_t10'])

# サプライズ5分位
df_surprise['surprise_q'] = pd.qcut(
    df_surprise['pre_car_5d'], q=5,
    labels=['Q1\n(低サプライズ)', 'Q2', 'Q3', 'Q4', 'Q5\n(高サプライズ)'],
    duplicates='drop'
)

print(f'サプライズ分析: {len(df_surprise)} イベント')
print(df_surprise.groupby('surprise_q')[['day0_ar', 'car_t1_t10', 'car_t11_t30']].mean().round(4))

In [ ]:
# サプライズ分位別 CAR time series
df_car_surp = df_car.merge(
    df_surprise[['symbol', 'earnings_date', 'surprise_q']],
    on=['symbol', 'earnings_date'], how='inner'
)

quintile_labels = df_surprise['surprise_q'].cat.categories.tolist()
quintile_colors = ['#d62728', '#ff7f0e', '#7f7f7f', '#1f77b4', '#2ca02c']

fig, ax = plt.subplots(figsize=(13, 6))

for q_label, color in zip(quintile_labels, quintile_colors):
    grp = df_car_surp[df_car_surp['surprise_q'] == q_label]
    car_mean = grp.groupby('t')['ar'].mean().cumsum() * 100
    ax.plot(car_mean.index, car_mean.values, label=q_label,
            color=color, linewidth=1.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.7, label='T=0（決算発表）')
ax.axvspan(1, 10, alpha=0.08, color='green')
ax.set_xlabel('イベント日 (T)')
ax.set_ylabel('CAR (%)')
ax.set_title('サプライズ分位別 CAR（T-10〜T+30）', fontsize=12)
ax.legend(title='サプライズ分位', fontsize=8, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_car_by_quintile.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 統計的検定: Q5 の T+1〜T+10 CAR は有意に正か？
q5_label = quintile_labels[-1]  # Q5
q5_car = df_surprise[df_surprise['surprise_q'] == q5_label]['car_t1_t10'].dropna()
q1_label = quintile_labels[0]   # Q1
q1_car = df_surprise[df_surprise['surprise_q'] == q1_label]['car_t1_t10'].dropna()

t_q5, p_q5 = ttest_1samp(q5_car.values, popmean=0)
t_q1, p_q1 = ttest_1samp(q1_car.values, popmean=0)

# ブートストラップ信頼区間
def bootstrap_ci(data, n_boot=N_BOOT, ci=0.95):
    boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    lo = (1 - ci) / 2 * 100
    hi = (1 + ci) / 2 * 100
    return np.percentile(boot_means, [lo, hi])

ci_q5 = bootstrap_ci(q5_car.values)
ci_q1 = bootstrap_ci(q1_car.values)

print('=== PEAD 統計的検定: T+1〜T+10 CAR ===')
print(f'Q5 (高サプライズ):')
print(f'  n={len(q5_car)}, mean={q5_car.mean():.4f}, t={t_q5:.3f}, p={p_q5:.4f}')
print(f'  95% CI: [{ci_q5[0]:.4f}, {ci_q5[1]:.4f}]')
print(f'  H₀ (CAR=0) 棄却: {"Yes" if p_q5 < ALPHA else "No"}  → PEAD 確認: {"Yes" if p_q5 < ALPHA and q5_car.mean() > 0 else "No"}')
print()
print(f'Q1 (低サプライズ):')
print(f'  n={len(q1_car)}, mean={q1_car.mean():.4f}, t={t_q1:.3f}, p={p_q1:.4f}')
print(f'  95% CI: [{ci_q1[0]:.4f}, {ci_q1[1]:.4f}]')
print(f'  H₀ (CAR=0) 棄却: {"Yes" if p_q1 < ALPHA else "No"}')

---
## 6. PEAD 持続性テスト

初期ドリフト（T+1〜T+5）が後期ドリフト（T+6〜T+20）を予測するか？

In [ ]:
# CAR 追加集計
persist_records = []
for (sym, ed), grp in df_car.groupby(['symbol', 'earnings_date']):
    car_early = grp[(grp['t'] >= 1) & (grp['t'] <= 5)]['ar'].sum()   # T+1〜T+5
    car_late  = grp[(grp['t'] >= 6) & (grp['t'] <= 20)]['ar'].sum()  # T+6〜T+20
    persist_records.append({
        'symbol': sym,
        'earnings_date': ed,
        'car_early': car_early,
        'car_late': car_late,
    })

df_persist = pd.DataFrame(persist_records).dropna()

# スピアマン IC
if len(df_persist) > 10:
    ic_persist, p_persist = spearmanr(df_persist['car_early'], df_persist['car_late'])
    t_persist = ic_persist * np.sqrt((len(df_persist) - 2) / (1 - ic_persist**2 + 1e-10))

    # 散布図
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(df_persist['car_early'] * 100, df_persist['car_late'] * 100,
               alpha=0.4, s=20, color='steelblue')
    # 回帰直線
    slope, intercept, r, p_lr, _ = stats.linregress(
        df_persist['car_early'], df_persist['car_late']
    )
    x_line = np.linspace(df_persist['car_early'].min(), df_persist['car_early'].max(), 100)
    ax.plot(x_line * 100, (intercept + slope * x_line) * 100, 'r-', linewidth=1.5,
            label=f'OLS (r={r:.3f}, p={p_lr:.3f})')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('CAR T+1〜T+5 (%)')
    ax.set_ylabel('CAR T+6〜T+20 (%)')
    ax.set_title(f'PEAD 持続性: 初期ドリフト → 後期ドリフト\nSpearman IC={ic_persist:.4f}, p={p_persist:.4f}',
                 fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '23_pead_persistence.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f'=== PEAD 持続性 IC ===')
    print(f'Spearman IC  : {ic_persist:+.4f}')
    print(f't 統計量     : {t_persist:+.4f}')
    print(f'p 値         : {p_persist:.4f}')
    print(f'PEAD 持続性  : {"Yes" if p_persist < ALPHA and ic_persist > 0 else "No"}')
else:
    print(f'サンプル不足 ({len(df_persist)} events) — 持続性分析をスキップ')

---
## 7. 決算期間のボラティリティ分析

決算前後 ±5 日の実現ボラティリティと通常時を比較する（ボラティリティ・クラッシュの確認）。

In [ ]:
# 各イベントのウィンドウ別ボラティリティ
vol_records = []

for (sym, ed), grp in df_car.groupby(['symbol', 'earnings_date']):
    grp = grp.dropna(subset=['ar'])
    # [-5, +5] ウィンドウ
    event_window = grp[grp['t'].between(-5, 5)]['ar']
    # [-30, -10] 通常期間（比較用）
    normal_window = grp[grp['t'].between(-30, -10)]['ar']
    if len(event_window) > 3 and len(normal_window) > 3:
        vol_records.append({
            'symbol': sym,
            'earnings_date': ed,
            'vol_event': event_window.std() * np.sqrt(252),    # 年率換算
            'vol_normal': normal_window.std() * np.sqrt(252),
        })

df_vol = pd.DataFrame(vol_records).dropna()
df_vol['vol_ratio'] = df_vol['vol_event'] / (df_vol['vol_normal'] + 1e-8)

print(f'=== 決算期間ボラティリティ比較 ===')
print(f'イベント数: {len(df_vol)}')
print(f'通常ボラ (年率)   : {df_vol["vol_normal"].mean():.4f} ({df_vol["vol_normal"].mean()*100:.1f}%)')
print(f'決算期間ボラ (年率): {df_vol["vol_event"].mean():.4f} ({df_vol["vol_event"].mean()*100:.1f}%)')
print(f'ボラ倍率          : {df_vol["vol_ratio"].mean():.2f}x')

# t 検定: 決算期間ボラ > 通常期間ボラ
t_vol, p_vol = stats.ttest_rel(df_vol['vol_event'], df_vol['vol_normal'])
print(f'対応ありt検定: t={t_vol:.3f}, p={p_vol:.4f}')
print(f'→ 決算期間のボラ有意に高い: {"Yes" if p_vol < ALPHA and t_vol > 0 else "No"}')

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ボックスプロット
ax = axes[0]
ax.boxplot(
    [df_vol['vol_normal'] * 100, df_vol['vol_event'] * 100],
    labels=['通常期間\n(T-30〜T-10)', '決算期間\n(T-5〜T+5)'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='navy'),
    medianprops=dict(color='red', linewidth=2)
)
ax.set_ylabel('実現ボラティリティ (年率, %)')
ax.set_title('決算期間 vs 通常期間のボラティリティ', fontsize=11)
ax.grid(alpha=0.3, axis='y')

# ボラ倍率分布
ax2 = axes[1]
ax2.hist(df_vol['vol_ratio'], bins=30, color='steelblue', edgecolor='white', alpha=0.8, density=True)
ax2.axvline(1.0, color='black', linewidth=1, linestyle='--', label='倍率=1.0（変化なし）')
ax2.axvline(df_vol['vol_ratio'].mean(), color='red', linewidth=1.5,
            label=f'平均倍率={df_vol["vol_ratio"].mean():.2f}x')
ax2.set_xlabel('ボラティリティ倍率 (決算/通常)')
ax2.set_ylabel('密度')
ax2.set_title('決算期間ボラティリティ倍率の分布', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.suptitle('決算期間のボラティリティ特性', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_earnings_volatility.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. トレーディングシグナルへの応用

決算ウィンドウを除外した場合と含む場合のモメンタム IC を比較し、  
「決算直前・直後 3 日間の除外」という標準的なフィルタの効果を定量評価する。

In [ ]:
# earnings_flag の作成: 各日が「決算発表前後3日以内」かどうかのフラグ
EARNINGS_WINDOW_DAYS = 3  # 前後 3 日

# 決算日を営業日カレンダーに対応させる
earnings_flag_set = set()  # (symbol, date) ペアのセット

for _, row in df_earnings.iterrows():
    sym = row['symbol']
    ed = pd.Timestamp(row['earnings_date'])

    for offset in range(-EARNINGS_WINDOW_DAYS, EARNINGS_WINDOW_DAYS + 1):
        d = get_date_at_offset(ed, offset, all_dates)
        if d is not None:
            earnings_flag_set.add((sym, d))

print(f'earnings_flag の対象 (symbol, date) ペア数: {len(earnings_flag_set):,}')

# OHLCV に earnings_flag を追加
df_ohlcv_flag = df_ohlcv.copy()
df_ohlcv_flag['earnings_flag'] = df_ohlcv_flag.apply(
    lambda r: (r['symbol'], r['date']) in earnings_flag_set, axis=1
)
print(f'earnings_flag=True の割合: {df_ohlcv_flag["earnings_flag"].mean():.1%}')

In [ ]:
# モメンタム因子の再計算（簡易版）
df_ohlcv_flag['mom_12m_minus_1m'] = df_ohlcv_flag.groupby('symbol')['close'].transform(
    lambda x: (np.log(x.shift(1) / x.shift(253)) - np.log(x.shift(1) / x.shift(22)))
)
df_ohlcv_flag['fwd_ret_5d'] = df_ohlcv_flag.groupby('symbol')['close'].transform(
    lambda x: x.pct_change(5).shift(-5)
)
df_ohlcv_flag['ym'] = df_ohlcv_flag['date'].dt.to_period('M')

# IC 計算: 除外なし vs 除外あり
def monthly_ic_filtered(df, factor_col, target_col, exclude_flag=None):
    df_f = df.copy()
    if exclude_flag:
        df_f = df_f[~df_f[exclude_flag]]
    df_month = df_f.sort_values('date').groupby(['symbol', 'ym']).last().reset_index()
    result = {}
    for ym, grp in df_month.groupby('ym'):
        sub = grp[[factor_col, target_col]].dropna()
        if len(sub) < 5:
            continue
        ic, _ = spearmanr(sub[factor_col], sub[target_col])
        result[ym] = ic
    return pd.Series(result)

ic_all     = monthly_ic_filtered(df_ohlcv_flag, 'mom_12m_minus_1m', 'fwd_ret_5d', exclude_flag=None)
ic_no_earn = monthly_ic_filtered(df_ohlcv_flag, 'mom_12m_minus_1m', 'fwd_ret_5d', exclude_flag='earnings_flag')
ic_all.index = ic_all.index.astype(str)
ic_no_earn.index = ic_no_earn.index.astype(str)

print('=== earnings_flag 除外効果 ===')
print(f'除外なし: IC 平均={ic_all.mean():+.4f}  std={ic_all.std():.4f}  ICIR={ic_all.mean()/ic_all.std():.3f}')
print(f'除外あり: IC 平均={ic_no_earn.mean():+.4f}  std={ic_no_earn.std():.4f}  ICIR={ic_no_earn.mean()/ic_no_earn.std():.3f}')

# 有意差検定
common_idx = ic_all.index.intersection(ic_no_earn.index)
if len(common_idx) > 5:
    t_diff, p_diff = stats.ttest_rel(ic_all.loc[common_idx].values, ic_no_earn.loc[common_idx].values)
    print(f'対応ありt検定 (除外なし vs 除外あり): t={t_diff:.3f}, p={p_diff:.4f}')

In [ ]:
# IC 比較チャート
common_idx = ic_all.index.intersection(ic_no_earn.index)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
x = np.arange(len(common_idx))
xtick_step = max(1, len(common_idx) // 12)

for ax, ic_series, label, color in [
    (axes[0], ic_all.loc[common_idx], '全データ (決算ウィンドウ含む)', '#d62728'),
    (axes[1], ic_no_earn.loc[common_idx], '決算前後3日除外', '#2ca02c'),
]:
    vals = ic_series.values
    bar_colors = [color if v > 0 else '#aaaaaa' for v in vals]
    ax.bar(x, vals, color=bar_colors, alpha=0.7, width=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    mean_ic = np.nanmean(vals)
    ax.axhline(mean_ic, color=color, linewidth=1.5, linestyle='--',
               label=f'Mean IC = {mean_ic:+.4f}')
    ax.set_ylabel('月次 IC')
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xticks(x[::xtick_step])
    ax.set_xticklabels(common_idx[::xtick_step], rotation=45, ha='right', fontsize=7)

plt.suptitle('モメンタム IC: 決算ウィンドウ除外の効果', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_ic_earnings_exclusion.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. 結論

### 仮説検定結果

| 仮説 | 内容 |
|------|------|
| **H₀** | 決算後 CAR = 0（PEAD なし） |
| **H₁** | Q5 の決算後 CAR > 0（PEAD あり）|
| **有意水準** | α = 0.05 |

### 主要結果
<!-- 実行後に記入 -->

| 指標 | 値 |
|------|----|
| Q5 CAR[T+1〜T+10] 平均 | — |
| Q5 t 値 | — |
| Q5 p 値 | — |
| PEAD 持続性 IC | — |
| 決算期間ボラ倍率 | — |
| earnings_flag 除外後 IC 改善 | — |

### 日本株での PEAD 証拠（Yes/No）
<!-- 記入 -->

### 推奨実装

1. **`earnings_date` を特徴量フラグとして追加する**
   - `earnings_flag`: 決算発表前後 3 営業日を示すバイナリ特徴量
   - モデルが決算ノイズを自動的に学習できる

2. **決算ウィンドウからのモメンタムシグナル除外**
   - 決算発表前後 3 日間はモメンタムシグナルを `NaN` に設定 → シグナル抑制
   - IC 比較で改善が確認できた場合のみ採用する

3. **PEAD 因子の特徴量化（追加検討）**
   - `car_t0_t2`: 決算発表後 2 日間の CAR をフォワードシグナルとして追加
   - 効果が確認できた場合、`42_equity_lgbm_walkforward.ipynb` の特徴量セットに組み込む

### 次のステップ
- `42_equity_lgbm_walkforward.ipynb` に `earnings_flag` を特徴量として追加する
- PEAD が有意な場合: `car_t0_t2` を追加特徴量として実験する

In [ ]:
# 最終サマリーの保存

# CAR サマリー
car_summary = mean_car[['t', 'mean', 'std', 'count', 'se', 'cum_ar']].copy()
car_summary.columns = ['t', 'mean_ar', 'std_ar', 'n_events', 'se_ar', 'cum_ar']
car_summary.to_csv(TABLES_DIR / '23_pead_car_summary.csv', index=False, encoding='utf-8-sig')

# サプライズ分位別サマリー
surprise_summary = df_surprise.groupby('surprise_q')[['day0_ar', 'car_t1_t10', 'car_t11_t30']].agg(
    ['mean', 'std', 'count']
)
surprise_summary.to_csv(TABLES_DIR / '23_pead_surprise_summary.csv', encoding='utf-8-sig')

# earnings_flag 付き OHLCV の保存（後続ノートブックで利用）
earnings_flag_df = df_ohlcv_flag[['symbol', 'date', 'earnings_flag']].copy()
earnings_flag_df.to_parquet(DATA_DIR / 'earnings_flag.parquet', index=False)

print('=== 保存完了 ===')
print(f'  {TABLES_DIR}/23_pead_car_summary.csv')
print(f'  {TABLES_DIR}/23_pead_surprise_summary.csv')
print(f'  {DATA_DIR}/earnings_flag.parquet')

print('\n=== 最終結果サマリー ===')
print('\n[CAR サマリー (T=-5〜+15)]')
print(car_summary[car_summary['t'].between(-5, 15)].to_string(index=False))

if 'ic_all' in dir() and 'ic_no_earn' in dir():
    print('\n[earnings_flag 除外効果]')
    print(f'除外なし IC: {ic_all.mean():+.4f}  ICIR: {ic_all.mean()/ic_all.std():.3f}')
    print(f'除外あり IC: {ic_no_earn.mean():+.4f}  ICIR: {ic_no_earn.mean()/ic_no_earn.std():.3f}')